# Voice-to-Text with Whisper

This notebook demonstrates how to use OpenAI's Whisper model for speech-to-text transcription in the VLA pipeline.

## Learning Objectives

By the end of this notebook, you will:
1. Understand audio capture fundamentals (16kHz PCM format)
2. Learn how Whisper processes speech into text
3. Implement voice activity detection (VAD)
4. Handle noise and improve transcription accuracy
5. Visualize audio waveforms and spectrograms

## Prerequisites

- Basic Python knowledge
- Understanding of NumPy arrays
- Microphone access (for live recording)

## Setup

In [ ]:
# Import required libraries
import sys
sys.path.append('../scripts')

import numpy as np
import matplotlib.pyplot as plt
from voice_interface import VoiceInterface, AudioConfig
import whisper
from pathlib import Path
import wave

# Configure matplotlib for better plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Part 1: Understanding Audio Fundamentals

### Sample Rate and PCM Format

Audio is stored as a sequence of samples:
- **Sample Rate**: Number of samples per second (Hz)
- **16kHz PCM**: 16,000 samples/second, 16-bit precision
- **Mono**: Single channel (vs. stereo = 2 channels)

Why 16kHz?
- Human speech frequency range: ~80-255 Hz (fundamental)
- Nyquist theorem: Sample rate ≥ 2× highest frequency
- 16kHz captures speech adequately while keeping file sizes small

In [ ]:
# Visualize how sample rate affects audio representation
duration = 0.01  # 10ms
freq = 440  # A4 note

# Different sample rates
rates = [8000, 16000, 44100]

fig, axes = plt.subplots(len(rates), 1, figsize=(12, 8))

for idx, rate in enumerate(rates):
    t = np.linspace(0, duration, int(rate * duration), endpoint=False)
    signal = np.sin(2 * np.pi * freq * t)
    
    axes[idx].plot(t * 1000, signal, 'o-', markersize=3)
    axes[idx].set_title(f'Sample Rate: {rate} Hz ({len(signal)} samples)')
    axes[idx].set_xlabel('Time (ms)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Higher sample rates capture more detail but increase data size")
print("16kHz is optimal for speech recognition")

## Part 2: Initialize Voice Interface

Whisper offers multiple model sizes with accuracy/speed tradeoffs:

| Model  | Parameters | Speed  | Accuracy | Use Case |
|--------|-----------|--------|----------|----------|
| tiny   | 39M       | ~32x   | Good     | Real-time, resource-constrained |
| base   | 74M       | ~16x   | Better   | **Recommended for VLA** |
| small  | 244M      | ~6x    | Great    | High accuracy needed |
| medium | 769M      | ~2x    | Excellent| Offline processing |
| large  | 1550M     | ~1x    | Best     | Maximum accuracy |

In [ ]:
# Initialize voice interface with base model
# This may take a moment to download the model on first run
vi = VoiceInterface(model_size="base")

print(f"Whisper model loaded: {vi.model_size}")
print(f"Device: {vi.model.device}")
print(f"Sample rate: {vi.config.sample_rate} Hz")

## Part 3: Audio Visualization

Let's create helper functions to visualize audio data.

In [ ]:
def plot_waveform(audio_data, sample_rate, title="Audio Waveform"):
    """
    Plot audio waveform showing amplitude over time.
    """
    duration = len(audio_data) / sample_rate
    time = np.linspace(0, duration, len(audio_data))
    
    plt.figure(figsize=(12, 4))
    plt.plot(time, audio_data, linewidth=0.5)
    plt.title(title)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Amplitude')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_spectrogram(audio_data, sample_rate, title="Spectrogram"):
    """
    Plot spectrogram showing frequency content over time.
    """
    plt.figure(figsize=(12, 6))
    plt.specgram(audio_data, Fs=sample_rate, cmap='viridis')
    plt.title(title)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Frequency (Hz)')
    plt.colorbar(label='Intensity (dB)')
    plt.tight_layout()
    plt.show()


def plot_rms_energy(audio_data, sample_rate, chunk_size=1024):
    """
    Plot RMS energy for voice activity detection visualization.
    """
    # Calculate RMS for each chunk
    num_chunks = len(audio_data) // chunk_size
    rms_values = []
    
    for i in range(num_chunks):
        chunk = audio_data[i*chunk_size:(i+1)*chunk_size]
        rms = np.sqrt(np.mean(chunk**2))
        rms_values.append(rms)
    
    time = np.arange(num_chunks) * chunk_size / sample_rate
    
    plt.figure(figsize=(12, 4))
    plt.plot(time, rms_values, linewidth=2)
    plt.axhline(y=0.01, color='r', linestyle='--', label='Silence Threshold')
    plt.title('RMS Energy (Voice Activity Detection)')
    plt.xlabel('Time (seconds)')
    plt.ylabel('RMS Energy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("Visualization functions loaded!")

## Part 4: Recording Audio

### Option 1: Live Recording (requires microphone)

In [ ]:
# UNCOMMENT TO RECORD LIVE AUDIO
# Note: This requires a working microphone

# print("Recording will start in 3 seconds...")
# import time
# time.sleep(3)

# Record 5 seconds of audio
# audio_data, sample_rate = vi.record_audio(duration=5.0, use_vad=False)

# print(f"Recording complete: {len(audio_data)/sample_rate:.2f} seconds")

# Visualize the recorded audio
# plot_waveform(audio_data, sample_rate, "Recorded Audio")
# plot_spectrogram(audio_data, sample_rate, "Recorded Audio Spectrogram")
# plot_rms_energy(audio_data, sample_rate)

### Option 2: Load Sample Audio File

For testing without a microphone, we'll create a synthetic voice command.

In [ ]:
# Generate synthetic audio for demonstration
# In practice, you would load a real audio file

sample_rate = 16000
duration = 3.0  # seconds
t = np.linspace(0, duration, int(sample_rate * duration))

# Create a simple "chirp" signal (frequency increases over time)
# This simulates the varying frequencies in speech
f0, f1 = 200, 1000  # Start and end frequencies
audio_data = np.sin(2 * np.pi * (f0 + (f1 - f0) * t / duration) * t)

# Add some noise to make it more realistic
noise = np.random.normal(0, 0.05, len(audio_data))
audio_data = audio_data * 0.5 + noise

print(f"Generated synthetic audio: {duration} seconds")

# Visualize
plot_waveform(audio_data, sample_rate, "Synthetic Audio Waveform")
plot_spectrogram(audio_data, sample_rate, "Synthetic Audio Spectrogram")
plot_rms_energy(audio_data, sample_rate)

## Part 5: Voice Activity Detection (VAD)

VAD detects when speech is present vs. silence. This is crucial for:
- Automatic recording start/stop
- Reducing processing of silent segments
- Improving user experience

### How VAD Works:
1. Calculate RMS (Root Mean Square) energy of audio chunks
2. Compare RMS to silence threshold
3. Track consecutive silent chunks
4. Stop recording after N silent chunks

In [ ]:
# Demonstrate VAD with synthetic data
# Create audio with speech -> silence -> speech pattern

sample_rate = 16000
chunk_size = 1024

# Speech segment (higher energy)
t1 = np.linspace(0, 1, sample_rate)
speech1 = 0.5 * np.sin(2 * np.pi * 400 * t1)

# Silence segment (low energy)
silence = np.random.normal(0, 0.01, sample_rate)

# Speech segment 2
t2 = np.linspace(0, 1, sample_rate)
speech2 = 0.5 * np.sin(2 * np.pi * 600 * t2)

# Combine
vad_audio = np.concatenate([speech1, silence, speech2])

# Plot with RMS energy
plot_waveform(vad_audio, sample_rate, "Audio with Speech-Silence-Speech Pattern")
plot_rms_energy(vad_audio, sample_rate)

print("Notice how RMS energy drops during silence!")
print("This is how VAD determines when to stop recording.")

## Part 6: Transcription with Whisper

Now let's transcribe audio using Whisper.

### How Whisper Works:
1. **Encoder**: Converts audio to embeddings (mel spectrogram → features)
2. **Decoder**: Generates text autoregressively (token by token)
3. **Language Detection**: Automatically identifies spoken language
4. **Timestamps**: Provides word-level timing information

In [ ]:
# For demonstration, let's create a sample audio file with actual speech
# In a real scenario, you would use the recorded audio

# Example: Transcribe a pre-recorded command
# You can replace this with your own audio file

sample_text = "navigate to the kitchen"
print(f"Expected transcription: '{sample_text}'")
print("\nNote: For actual transcription, you need a real audio file.")
print("The synthetic audio above won't produce meaningful text.")

# UNCOMMENT to transcribe real audio:
# result = vi.transcribe(audio_data, language="en")
# print(f"\nTranscription: {result['text']}")
# print(f"Language: {result['language']}")
# print(f"Confidence: {result['confidence']:.2f}")

## Part 7: Handling Different Languages

In [ ]:
# Whisper supports 99 languages!
# Common examples:

languages = {
    "en": "English",
    "es": "Spanish",
    "fr": "French",
    "de": "German",
    "zh": "Chinese",
    "ja": "Japanese",
    "ar": "Arabic",
    "hi": "Hindi"
}

print("Whisper Supported Languages (sample):")
for code, name in languages.items():
    print(f"  {code}: {name}")

print("\nTo transcribe in a specific language:")
print("  result = vi.transcribe(audio_data, language='es')")
print("\nTo auto-detect language:")
print("  result = vi.transcribe(audio_data, language=None)")

## Part 8: Noise Handling

Real-world audio often contains noise. Let's see how Whisper handles it.

In [ ]:
# Create audio with varying noise levels
duration = 2.0
t = np.linspace(0, duration, int(sample_rate * duration))
clean_signal = 0.5 * np.sin(2 * np.pi * 440 * t)

noise_levels = [0.0, 0.1, 0.3, 0.5]

fig, axes = plt.subplots(len(noise_levels), 1, figsize=(12, 10))

for idx, noise_level in enumerate(noise_levels):
    noise = np.random.normal(0, noise_level, len(clean_signal))
    noisy_signal = clean_signal + noise
    
    axes[idx].plot(t[:1000], noisy_signal[:1000], linewidth=1)
    axes[idx].set_title(f'Noise Level: {noise_level} (SNR ≈ {-20*np.log10(noise_level) if noise_level > 0 else "∞":.1f} dB)')
    axes[idx].set_xlabel('Time (s)')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Tips for handling noise:")
print("1. Use a good quality microphone")
print("2. Record in a quiet environment")
print("3. Position microphone close to speaker (15-30cm)")
print("4. Use larger Whisper models for better noise robustness")
print("5. Consider audio preprocessing (noise reduction filters)")

## Part 9: Complete Example - Voice Command Pipeline

In [ ]:
def process_voice_command(vi, duration=5.0, visualize=True):
    """
    Complete voice command processing pipeline.
    
    Steps:
    1. Record audio with VAD
    2. Visualize waveform and spectrogram
    3. Transcribe using Whisper
    4. Return transcribed command
    """
    print("=" * 60)
    print("Voice Command Pipeline")
    print("=" * 60)
    
    # Step 1: Record
    print("\n[1/3] Recording audio...")
    # UNCOMMENT for live recording:
    # audio_data, sr = vi.record_audio(duration=duration, use_vad=True)
    
    # For demo, use synthetic data
    print("(Using synthetic audio for demo)")
    sr = 16000
    t = np.linspace(0, duration, int(sr * duration))
    audio_data = 0.3 * np.sin(2 * np.pi * 440 * t)
    
    # Step 2: Visualize
    if visualize:
        print("\n[2/3] Visualizing audio...")
        plot_waveform(audio_data, sr)
        plot_rms_energy(audio_data, sr)
    
    # Step 3: Transcribe
    print("\n[3/3] Transcribing...")
    # UNCOMMENT for real transcription:
    # result = vi.transcribe(audio_data, language="en")
    # command = result['text']
    # confidence = result['confidence']
    
    # Demo placeholder
    command = "(synthetic audio - no meaningful transcription)"
    confidence = 0.0
    
    print("\n" + "=" * 60)
    print(f"Command: {command}")
    print(f"Confidence: {confidence:.2f}")
    print("=" * 60)
    
    return command

# Run the pipeline
# UNCOMMENT to test with real audio:
# command = process_voice_command(vi, duration=5.0)

print("Pipeline function defined! Uncomment to run with real audio.")

## Exercise: Transcribe Your Own Voice Command

### Task:
1. Uncomment the live recording code above
2. Record yourself saying: "Navigate to the kitchen and pick up the red cup"
3. Visualize the audio waveform and spectrogram
4. Transcribe the command using Whisper
5. Compare transcription accuracy with different model sizes (tiny vs. base vs. small)

### Questions to Explore:
1. How does background noise affect transcription accuracy?
2. What is the tradeoff between model size and transcription speed?
3. How does confidence score correlate with actual accuracy?
4. Can Whisper handle commands with technical robotics terms ("grasp", "navigate", "waypoint")?

In [ ]:
# YOUR CODE HERE
# Try the exercise above!

## Summary

In this notebook, you learned:

✅ **Audio Fundamentals**
- 16kHz PCM format for speech
- Sample rate importance
- Mono vs. stereo

✅ **Voice Activity Detection (VAD)**
- RMS energy calculation
- Silence threshold detection
- Automatic recording control

✅ **Whisper Transcription**
- Model size selection
- Language detection
- Confidence scoring

✅ **Audio Visualization**
- Waveform plots
- Spectrograms
- Energy analysis

✅ **Noise Handling**
- Best practices for clean audio
- Noise robustness strategies

### Next Steps:
- Proceed to `02_llm_planning.ipynb` to learn how to convert transcribed text into robot action plans
- Experiment with different Whisper models
- Try transcribing commands in different languages
- Integrate voice interface with your robot system